# 0. Setup Environment

In [ ]:
# Competition data input
COMP_INPUT = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"

# Your uploaded models dataset
MODEL_PKL = "/kaggle/input/datasets/brandonkhuu/rogii-xgb-baseline/fold_models.pkl"

# === Usually no edits needed below ===
from pathlib import Path  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

TEST_DIR = Path(COMP_INPUT) / "test"
SAMPLE_SUB_PATH = Path(COMP_INPUT) / "sample_submission.csv"
OUTPUT_PATH = Path("/kaggle/working/submission.csv")

# Schema constants
COL_MD, COL_X, COL_Y, COL_Z = "MD", "X", "Y", "Z"
COL_GR = "GR"
COL_TVT_INPUT = "TVT_input"
COL_TVT = "TVT"

# Feature columns — MUST match what the model was trained on (config: configs/xgb_baseline.yaml)
FEATURE_COLS = [
    "md",
    "x",
    "y",
    "z",
    "gr",
    "gr_local_mean_50",
    "gr_local_std_50",
    "anchor_tvt",
    "anchor_z",
    "anchor_md",
    "anchor_gr",
    "smooth_anchor_tvt_50",
    "slope_tvt_md_50",
    "d_md_from_anchor",
    "d_z_from_anchor",
    "d_gr_from_anchor",
    "gr_match_tvt",
    "gr_match_sim",
    "gr_match_delta_anchor",
]
ANCHOR_COL = "anchor_tvt"  # used for residual -> absolute conversion

print(f"Test dir:        {TEST_DIR} (exists: {TEST_DIR.exists()})")
print(f"Sample sub:      {SAMPLE_SUB_PATH} (exists: {SAMPLE_SUB_PATH.exists()})")
print(f"Models pickle:   {MODEL_PKL} (exists: {Path(MODEL_PKL).exists()})")

# 1. Featurizer

Same code as the local pipeline. Two functions: `predict_gr_match_v2` (typewell GR-template
correlation) and `featurize_well` (computes the 19 features per eval row).

In [ ]:
def predict_gr_match_v2(g, tw, window=51, search_radius=50.0, smooth=11):
    """For each eval row, find the typewell TVT whose GR window best matches.
    Returns (pred, sim) — both arrays of length len(g)."""
    n = len(g)
    pred = g[COL_TVT_INPUT].values.astype(float).copy()
    sim = np.zeros(n, dtype=float)
    eval_mask = np.isnan(pred)
    if not eval_mask.any():
        return pred, sim
    known_idx = np.where(~eval_mask)[0]
    if len(known_idx) == 0:
        return pred, sim
    anchor = float(pred[known_idx[-1]])
    eval_idx = np.where(eval_mask)[0]

    lat_gr = pd.Series(g[COL_GR].values).interpolate(limit_direction="both").values.astype(float)
    if not np.isfinite(lat_gr).all():
        pred[eval_mask] = anchor
        return pred, sim
    lat_gr_z = (lat_gr - lat_gr.mean()) / (lat_gr.std() + 1e-9)

    if tw is None or COL_GR not in tw or COL_TVT not in tw:
        pred[eval_mask] = anchor
        return pred, sim
    tw_clean = (
        tw.dropna(subset=[COL_TVT, COL_GR]).drop_duplicates(subset=[COL_TVT]).sort_values(COL_TVT)
    )
    if len(tw_clean) < window + 2:
        pred[eval_mask] = anchor
        return pred, sim
    tvt_grid = np.arange(
        np.floor(tw_clean[COL_TVT].min()), np.ceil(tw_clean[COL_TVT].max()) + 1, 1.0
    )
    if len(tvt_grid) < window + 2:
        pred[eval_mask] = anchor
        return pred, sim
    gr_grid = np.interp(tvt_grid, tw_clean[COL_TVT].values, tw_clean[COL_GR].values)
    gr_grid_z = (gr_grid - gr_grid.mean()) / (gr_grid.std() + 1e-9)

    half = window // 2
    n_tw = len(gr_grid_z)
    TW_W = np.lib.stride_tricks.sliding_window_view(gr_grid_z, window).copy()
    norms = np.linalg.norm(TW_W, axis=1, keepdims=True)
    norms[norms < 1e-9] = 1.0
    TW_W /= norms
    tw_centers = tvt_grid[half : n_tw - half]

    cand_mask = np.abs(tw_centers - anchor) <= search_radius
    if not cand_mask.any():
        pred[eval_mask] = anchor
        return pred, sim

    n_lat = len(lat_gr_z)
    LAT_W_all = np.lib.stride_tricks.sliding_window_view(lat_gr_z, window)
    in_bounds = (eval_idx >= half) & (eval_idx <= n_lat - half - 1)
    pred[eval_idx[~in_bounds]] = anchor
    eval_full = eval_idx[in_bounds]
    if len(eval_full) == 0:
        return pred, sim

    LAT_W = LAT_W_all[eval_full - half].copy()
    lat_norms = np.linalg.norm(LAT_W, axis=1, keepdims=True)
    lat_norms[lat_norms < 1e-9] = 1.0
    LAT_W /= lat_norms

    S = LAT_W @ TW_W.T
    S_masked = S.copy()
    S_masked[:, ~cand_mask] = -np.inf
    best = np.argmax(S_masked, axis=1)
    raw = tw_centers[best]
    raw_sim = S[np.arange(len(eval_full)), best]

    if smooth and smooth > 1:
        raw = pd.Series(raw).rolling(smooth, center=True, min_periods=1).median().values

    pred[eval_full] = raw
    sim[eval_full] = raw_sim
    return pred, sim


def featurize_well(g, tw, well_id=None):
    """Compute features for the eval rows of one well."""
    g = g.reset_index(drop=True)
    n = len(g)
    eval_mask = g[COL_TVT_INPUT].isna().values
    if not eval_mask.any():
        return pd.DataFrame(columns=["well", "row_idx", *FEATURE_COLS])
    known_idx = np.where(~eval_mask)[0]
    if len(known_idx) == 0:
        return pd.DataFrame(columns=["well", "row_idx", *FEATURE_COLS])

    anchor_idx = known_idx[-1]
    anchor_tvt = float(g[COL_TVT_INPUT].iloc[anchor_idx])
    anchor_z = float(g[COL_Z].iloc[anchor_idx])
    anchor_md = float(g[COL_MD].iloc[anchor_idx])
    gr_filled = pd.Series(g[COL_GR].values).interpolate(limit_direction="both").fillna(0.0).values
    anchor_gr = float(gr_filled[anchor_idx])

    K = 50
    last_K = known_idx[-K:] if len(known_idx) >= K else known_idx
    smooth_anchor = float(g[COL_TVT_INPUT].iloc[last_K].mean())
    slope = (
        float(np.polyfit(g[COL_MD].values[last_K], g[COL_TVT_INPUT].values[last_K], 1)[0])
        if len(last_K) >= 2
        else 0.0
    )

    gr_series = pd.Series(gr_filled)
    gr_local_mean = gr_series.rolling(50, center=True, min_periods=1).mean().values
    gr_local_std = gr_series.rolling(50, center=True, min_periods=1).std().fillna(0.0).values

    gr_match_pred, gr_match_sim = predict_gr_match_v2(g, tw)

    feats_full = pd.DataFrame(
        {
            "md": g[COL_MD].values,
            "x": g[COL_X].values,
            "y": g[COL_Y].values,
            "z": g[COL_Z].values,
            "gr": gr_filled,
            "gr_local_mean_50": gr_local_mean,
            "gr_local_std_50": gr_local_std,
            "anchor_tvt": anchor_tvt,
            "anchor_z": anchor_z,
            "anchor_md": anchor_md,
            "anchor_gr": anchor_gr,
            "smooth_anchor_tvt_50": smooth_anchor,
            "slope_tvt_md_50": slope,
            "d_md_from_anchor": g[COL_MD].values - anchor_md,
            "d_z_from_anchor": g[COL_Z].values - anchor_z,
            "d_gr_from_anchor": gr_filled - anchor_gr,
            "gr_match_tvt": gr_match_pred,
            "gr_match_sim": gr_match_sim,
            "gr_match_delta_anchor": gr_match_pred - anchor_tvt,
        }
    )
    feats_full["well"] = (
        well_id if well_id is not None else (g["well"].iloc[0] if "well" in g.columns else None)
    )
    feats_full["row_idx"] = np.arange(n, dtype=np.int32)

    out = feats_full.loc[eval_mask].reset_index(drop=True)
    return out[["well", "row_idx", *FEATURE_COLS]]

# 2. Load Test Wells and Featurize

In [ ]:
test_rows = []
test_files = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
print(f"Found {len(test_files)} test horizontal-well CSVs")

for f in test_files:
    well = f.name.replace("__horizontal_well.csv", "")
    g = pd.read_csv(f)
    g["well"] = well
    tw_path = TEST_DIR / f"{well}__typewell.csv"
    tw = pd.read_csv(tw_path) if tw_path.exists() else None
    test_rows.append(featurize_well(g, tw, well))
    print(f"  {well}: total rows in CSV = {len(g)}, eval rows = {len(test_rows[-1])}")

test_feats = pd.concat(test_rows, ignore_index=True)
for c in test_feats.columns:
    if test_feats[c].dtype == "float64":
        test_feats[c] = test_feats[c].astype("float32")

print(f"\nTotal featurized eval rows: {len(test_feats):,}")

# 3. Load Fold Models

In [ ]:
fold_models = pd.read_pickle(MODEL_PKL)
print(f"Loaded {len(fold_models)} fold models")
for i, m in enumerate(fold_models):
    bi = getattr(m, "best_iteration", None)
    print(f"  fold {i}: best_iteration = {bi}")

# 4. Predict and write submission

In [ ]:
X_test = test_feats[FEATURE_COLS].astype("float32").values
residual_preds = np.mean([m.predict(X_test) for m in fold_models], axis=0)
tvt_preds = residual_preds + test_feats[ANCHOR_COL].astype("float64").values

ss = pd.read_csv(SAMPLE_SUB_PATH)
id_col, val_col = ss.columns[0], ss.columns[1]
print(f"sample_submission cols: id={id_col!r}, value={val_col!r}, rows={len(ss):,}")

pred_dict = {
    f"{w}_{int(r)}": float(p)
    for w, r, p in zip(
        test_feats["well"].values, test_feats["row_idx"].values, tvt_preds, strict=False
    )
}

out = ss.copy()
out[val_col] = out[id_col].map(pred_dict)
n_missing = int(out[val_col].isna().sum())
print(f"\nPredictions missing for {n_missing} of {len(out)} sample_submission rows")
assert n_missing == 0, "mismatch — check well/row_idx alignment vs sample_submission"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved: {OUTPUT_PATH}")
out.head()